# Extractor Agent

This notebook extracts atomic clinical facts from original MTSamples notes. It does not simplify or interpret the notes. Each atom includes the exact source text and a category for downstream agents.

## Output format

```json
{
  "atoms": [
    {
      "text": "denies chest pain",
      "type": "negative_finding",
      "source_quote": "denies chest pain"
    }
  ],
  "not_found_types": ["diagnosis"]
}
```

In [7]:
import json
import os
import re
from pathlib import Path
from typing import Any

import pandas as pd

try:
    from google import genai
    from google.genai import types
except ImportError:
    genai = None
    types = None

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "mtsamples_baseline_outputs.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "extractor_outputs.jsonl"
MODEL = "gemini-3.5-flash"

ALLOWED_TYPES = {
    "diagnosis", "diagnosis_reasoning", "symptom", "risk", "medical_history",
    "family_history", "environmental_history", "demographic", "medical_specialty",
    "examination", "examination_finding", "lab_value", "vital_sign", "procedure",
    "treatment", "medication", "referral", "follow_up", "patient_education",
}
DATA_PATH

WindowsPath('C:/Users/aliam/OneDrive/Desktop/Google 2A/Google-2A-faithfulmed/data/mtsamples_baseline_outputs.csv')

In [ ]:
EXTRACTOR_INSTRUCTIONS = """
You extract atomic clinical facts from a clinical note.

The input is JSON containing clinical_note. Identify only what the source says.
Do not rewrite, simplify, interpret, diagnose, explain, infer, or add information.

Return JSON with exactly these fields:
{
  "atoms": [
    {"text": "...", "type": "...", "source_quote": "..."}
  ],
  "not_found_types": []
}

For each atom:
- text: exact words copied from the clinical note. These should be complete sentences.
- type: exactly one allowed category.
- source_quote: exact source text supporting the atom.
If the atom is extracted from under a heading, include the source text from the heading up to wherever needed

Keep each atom atomic: one finding, symptom, medication, lab value, procedure,
or instruction per atom. Preserve spelling, abbreviations, typos, numbers, units,
doses, routes, frequencies, negations, qualifiers, status, and timing.

Do not change past, stopped, resolved, suspected, negated, planned, or
recommended information into current or confirmed information. If status or
temporality is stated, keep it in the copied text. If it is not stated, do not
guess it.

Allowed categories:
- diagnosis: explicit diagnoses, impressions, assessments, or conditions.
- diagnosis_reasoning: explicit rationale or findings supporting a diagnosis.
- symptom: reported or observed symptoms and complaints.
- risk: explicitly stated risks or risk factors.
- medical_history: past medical, surgical, or patient history.
- family_history: health information about family members.
- environmental_history: social, occupational, living, substance-use, travel,
  or exposure history.
- demographic: explicitly stated age, sex, gender, pronouns,
  race, ethnicity, cultural identity, language preference, or relevant context.
- medical_specialty: specialty or department explicitly named in the note.
- examination: examinations, tests, scans, or physical exams performed or ordered.
- examination_finding: a specific finding from an examination or test.
- lab_value: laboratory or measured result with its value and units.
- vital_sign: blood pressure, heart rate, temperature, respiratory rate, oxygen
  saturation, height, weight, BMI, or another reported vital sign.
- procedure: procedure or operation performed, ordered, planned, or recommended.
- treatment: therapy or management other than a named medication.
- medication currently being taken or used, medication prescribed or recommended, past medication.
- referral: referral to another clinician, service, department, or specialty.
- follow_up: follow-up, return instruction, monitoring plan, or future appointment.
- patient_education: counseling, instructions, or educational resources given.

Rules:
1. Use only the clinical note. Do not use outside knowledge.
2. Do not expand abbreviations, correct typos, paraphrase, or normalize text.
3. Do not infer identity, ethnicity, culture, religion, language, gender,
   nationality, specialty, diagnosis, risk, or reasoning.
4. Do not extract names, dates of birth, addresses, phone numbers, or other
   direct identifiers unless the task explicitly requires them.
5. Every atom must have source text copied from the note.
6. Do not create atoms with text such as None, not found, or unknown.
7. List every category with no supporting text in not_found_types.
8. Return only JSON.
9. If there are no facts, return {{"atoms": [], "not_found_types": [...] }}.
10. Refer to the patient as "patient"

The clinical note is provided in the input JSON as clinical_note.
"""

RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "atoms": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "text": {"type": "STRING"},
                    "type": {"type": "STRING", "enum": sorted(ALLOWED_TYPES)},
                    "source_quote": {"type": "STRING"},
                },
                "required": ["text", "type", "source_quote"],
            },
        },
        "not_found_types": {
            "type": "ARRAY",
            "items": {"type": "STRING", "enum": sorted(ALLOWED_TYPES)},
        },
    },
    "required": ["atoms", "not_found_types"],
}


In [9]:
def normalize_whitespace(value: str) -> str:
    return re.sub(r"\s+", " ", value).strip()


def quote_is_in_source(source_text: str, source_quote: str) -> bool:
    if not isinstance(source_text, str) or not isinstance(source_quote, str):
        return False
    if not source_quote.strip():
        return False
    return source_quote in source_text or normalize_whitespace(source_quote) in normalize_whitespace(source_text)


def validate_output(model_output: dict[str, Any], source_text: str) -> dict[str, Any]:
    if not isinstance(model_output, dict):
        print("Extractor response must be a JSON object.")
    atoms = model_output.get("atoms")
    not_found_types = model_output.get("not_found_types")
    if not isinstance(atoms, list) or not isinstance(not_found_types, list):
        print("Extractor response must contain atoms and not_found_types lists.")
    invalid_not_found = set(not_found_types) - ALLOWED_TYPES
    if invalid_not_found:
        print(f"Invalid not_found_types: {sorted(invalid_not_found)}")
    validated_atoms = []
    for index, atom in enumerate(atoms):
        if not isinstance(atom, dict):
            print(f"Atom {index} is not an object.")
        required = {"text", "type", "source_quote"}
        missing = required - atom.keys()
        if missing:
            print(f"Atom {index} is missing fields: {sorted(missing)}")
        if atom["type"] not in ALLOWED_TYPES:
            print(f"Atom {index} has invalid type: {atom['type']}")
        if not all(isinstance(atom[field], str) and atom[field].strip() for field in required):
            print(f"Atom {index} has an empty or non-string field.")
        if not quote_is_in_source(source_text, atom["text"]):
            print(f"Atom {index} text was not found in the source text.")
        if not quote_is_in_source(source_text, atom["source_quote"]):
            print(f"Atom {index} source_quote was not found in the source text.")
        validated_atoms.append({field: atom[field].strip() for field in required})
    present_types = {atom["type"] for atom in validated_atoms}
    expected_not_found = ALLOWED_TYPES - present_types
    if set(not_found_types) != expected_not_found:
        print("not_found_types must contain exactly the categories absent from atoms.")
    return {"atoms": validated_atoms, "not_found_types": sorted(not_found_types)}


In [10]:
records = pd.read_csv(DATA_PATH)
required_columns = {"sample_name", "transcription"}
missing_columns = required_columns - set(records.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
records = records[["sample_name", "transcription"]].dropna()
records = records[records["transcription"].str.strip().ne("")].reset_index(drop=True)
print(f"Loaded {len(records):,} original MTSamples notes.")
print(records[["sample_name"]].head())

Loaded 50 original MTSamples notes.
                                      sample_name
0   Bariatric Consult - Surgical Weight Loss - 4 
1                  Consult - Atrial Fibrillation 
2              Consult - Atrial Fibrillation - 1 
3                     Allergy Evaluation Consult 
4                       Cardiac Consultation - 2 


In [11]:
def make_client():
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise RuntimeError("Set GEMINI_API_KEY before running Gemini extraction.")
    if genai is None or types is None:
        raise ImportError("Install the existing google-genai dependency first.")
    return genai.Client(api_key=api_key)


def extract_atoms(source_text: str, client: Any | None = None) -> dict[str, Any]:
    if client is None:
        client = make_client()
    prompt = EXTRACTOR_INSTRUCTIONS + "\n\nINPUT JSON:\n" + json.dumps({"clinical_note": source_text})
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RESPONSE_SCHEMA,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        ),
    )
    if not response.text:
        raise ValueError("Gemini returned an empty response.")
    model_output = json.loads(response.text)
    return validate_output(model_output, source_text)


In [12]:
RUN_REAL_EXTRACTION = True
sample_source = records.iloc[0]["transcription"]
if RUN_REAL_EXTRACTION:
    sample_output = extract_atoms(sample_source)
    print(json.dumps(sample_output, indent=2))

Atom 64 text was not found in the source text.
Atom 64 source_quote was not found in the source text.
{
  "atoms": [
    {
      "text": "Ms. A is a 55-year-old female",
      "type": "demographic",
      "source_quote": "Ms. A is a 55-year-old female"
    },
    {
      "text": "presented to the Bariatric Surgery Service",
      "type": "medical_specialty",
      "source_quote": "presented to the Bariatric Surgery Service"
    },
    {
      "text": "for consideration of laparoscopic Roux-en-Y gastric bypass.",
      "type": "procedure",
      "source_quote": "for consideration of laparoscopic Roux-en-Y gastric bypass."
    },
    {
      "text": "The patient states that she has been overweight for approximately 35 years",
      "type": "medical_history",
      "source_quote": "The patient states that she has been overweight for approximately 35 years"
    },
    {
      "text": "and has tried multiple weight loss modalities in the past including Weight Watchers, NutriSystem, Jenny Cr

In [13]:
def run_extraction(records_to_process: pd.DataFrame, output_path: Path, limit: int = 5) -> None:
    client = make_client()
    selected = records_to_process.head(limit)
    with output_path.open("w", encoding="utf-8") as output_file:
        for row in selected.itertuples(index=False):
            extraction = extract_atoms(row.transcription, client=client)
            output_file.write(json.dumps({
                "sample_name": row.sample_name,
                **extraction,
            }) + "\n")
    print(f"Saved {len(selected)} extraction results to {output_path}")


# Uncomment after testing one real note.
# run_extraction(records, OUTPUT_PATH, limit=5)